# Clarillo Predice: riesgo de incendio en el entorno del Parque Nacional Río Clarillo

**Actividad A+S - Taller de Analítica Predictiva y Prescriptiva (IDA602)**
**Estudiante:** Alejandro Espinoza
**Socio comunitario:** guardaparque del Parque Nacional Río Clarillo (CONAF), comuna de Pirque

Este notebook reúne los datos y los modelos de la etapa de ejecución. Responde a los tres objetivos del proyecto:

| Objetivo | Meta | Sección |
|---|---|---|
| 1. Identificar las variables que explican los incendios en el entorno de Pirque | ranking de al menos 10 variables | 5 |
| 2. Modelos que estimen cuándo y dónde es más probable un incendio | AUC ≥ 0,75 en prueba, superar las reglas de CONAF y un mapa cuyo 20% del área concentre al menos el 50% de los incendios | 3, 4 y 6 |
| 3. Boletín semanal con IA generativa y pronóstico de 7 días | boletín por correo para el equipo del parque | 7 |

Los datos se cargan desde GitHub, por lo que el notebook se puede ejecutar completo en Google Colab sin archivos locales.

In [ ]:
# Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score,
                             f1_score, precision_recall_curve, RocCurveDisplay)

## 1. Datos

Los datos se prepararon en la primera parte de la ejecución, a partir de fuentes públicas:

- **Incendios CONAF** 2002-03 a 2024-25: registro histórico de Datos para Resiliencia (2002-2020) y capas SIMEF (2020-2025), con fecha y coordenadas de cada incendio.
- **Clima Open-Meteo** (reanálisis ERA5) diario y por hora, en un punto cercano al parque. Se validó con la estación DGA de Pirque.
- **Índices de CONAF** calculados con el clima de la tarde: humedad del combustible fino muerto (HCFM), probabilidad de ignición (PI), regla 30-30-30 y una aproximación del Botón Rojo.
- **Territorio** en celdas de 2 x 2 km: relieve (SRTM), vegetación (NDVI MODIS), caminos y edificaciones (OpenStreetMap).

Se usan tres archivos:

| Archivo | Contenido |
|---|---|
| `dataset_incendios_pirque_diario.csv` | un día por fila (8.240 días), con la variable objetivo y 36 variables |
| `incendios_entorno.csv` | los 888 incendios a 25 km o menos del parque, con fecha, causa y celda |
| `celdas_entorno.csv` | las 484 celdas de 2 x 2 km del entorno, con sus características |

In [ ]:
url_base = "https://raw.githubusercontent.com/alejandro7espinoza-star/clarillo-predice/main/"

df = pd.read_csv(url_base + "dataset_incendios_pirque_diario.csv", parse_dates=["fecha"])
incendios = pd.read_csv(url_base + "incendios_entorno.csv", parse_dates=["fecha"])
celdas = pd.read_csv(url_base + "celdas_entorno.csv")

print("Días:", df.shape, "| desde", df["fecha"].min().date(), "hasta", df["fecha"].max().date())
print("Incendios en el entorno:", len(incendios))
print("Celdas:", len(celdas))
print("Nulos en el dataset diario:", df.isna().sum().sum())

**Variable objetivo:** `dia_critico` = 1 si ese día se inició al menos un incendio a 25 km o menos del parque.

Las columnas `n_entorno`, `dia_incendio_pirque`, `n_pirque`, `n_rm` y `sup_entorno_ha` cuentan los incendios del mismo día. No se usan como predictoras, porque el modelo conocería la respuesta.

## 2. Análisis exploratorio

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(17, 4))

(df.groupby("mes")["dia_critico"].mean() * 100).plot(kind="bar", ax=ejes[0], color="#4C72B0")
ejes[0].set_title("% de días críticos por mes")
ejes[0].set_xlabel("Mes")
ejes[0].tick_params(axis="x", rotation=0)

incendios.groupby("temporada").size().plot(ax=ejes[1], marker="o")
ejes[1].set_title("Incendios por temporada en el entorno")
ejes[1].tick_params(axis="x", rotation=90)

incendios["causa_grupo"].value_counts().head(6).iloc[::-1].plot(kind="barh", ax=ejes[2], color="#4C72B0")
ejes[2].set_title("Causas más frecuentes")
plt.tight_layout()
plt.show()

print("Días críticos:", round(df["dia_critico"].mean() * 100, 1), "%")
print("Incendios entre noviembre y marzo:", round(incendios["fecha"].dt.month.isin([11, 12, 1, 2, 3]).mean() * 100, 1), "%")
print("Promedio por temporada 2002-2012:", round(incendios[incendios["temporada"] <= "2011-2012"].groupby("temporada").size().mean(), 1))
print("Promedio por temporada 2022-2025:", round(incendios[incendios["temporada"] >= "2022-2023"].groupby("temporada").size().mean(), 1))

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(celdas["lon"], celdas["lat"], c="lightgray", marker="s", s=35, label="celdas de 2 x 2 km")
plt.scatter(incendios["lon"], incendios["lat"], s=6, c="red", label="incendios 2002-2025")
plt.scatter([-70.45], [-33.75], marker="*", s=300, c="green", edgecolor="k", label="PN Río Clarillo")
plt.title("Incendios en el entorno de 25 km del parque")
plt.legend(loc="lower left")
plt.show()

**Observaciones**

- Solo el 9,1% de los días son críticos: las clases están desbalanceadas, por lo que se usan AUC, PR AUC, recall y F1 en vez de accuracy.
- El 90% de los incendios ocurre entre noviembre y marzo.
- Los incendios por temporada se duplicaron: de 32 en promedio entre 2002 y 2012 a 71 en las tres últimas temporadas.
- La mayoría tiene origen humano (tránsito de personas o vehículos e intencionales) y se concentra al noroeste del parque, cerca de Puente Alto y Pirque.

## 3. Modelo de días críticos (¿cuándo?)

**Partición temporal:** se entrena con las temporadas 2002-03 a 2021-22 y se evalúa con 2022-23 a 2024-25, como se usaría en la realidad. No se mezclan días al azar.

**Modelo:** Random Forest. Se eligió comparando regresión logística, Random Forest y Gradient Boosting, y se optimizó con búsqueda de hiperparámetros (`GridSearchCV`) y validación cruzada temporal (`TimeSeriesSplit`). Aquí se usan directamente los parámetros encontrados.

**Variables agregadas:** tasa histórica de días críticos del mes (solo con entrenamiento) y el mes como ciclo (seno y coseno), para que diciembre y enero queden cerca.

In [ ]:
fuga = ["n_entorno", "dia_incendio_pirque", "n_pirque", "n_rm", "sup_entorno_ha"]
datos = df.drop(columns=fuga).sort_values("fecha").reset_index(drop=True)

train = datos[datos["temporada"] <= "2021-2022"]
tasa_mes = train.groupby("mes")["dia_critico"].mean()
datos["tasa_mes"] = datos["mes"].map(tasa_mes)
datos["mes_sin"] = np.sin(2 * np.pi * datos["mes"] / 12)
datos["mes_cos"] = np.cos(2 * np.pi * datos["mes"] / 12)

variables = [c for c in datos.columns if c not in ["fecha", "temporada", "dia_critico"]]
train = datos[datos["temporada"] <= "2021-2022"]
test = datos[datos["temporada"] >= "2022-2023"]
X_train, y_train = train[variables], train["dia_critico"]
X_test, y_test = test[variables], test["dia_critico"]

print("Variables predictoras:", len(variables))
print("Entrenamiento:", X_train.shape, "| días críticos:", round(y_train.mean(), 3))
print("Prueba:", X_test.shape, "| días críticos:", round(y_test.mean(), 3))

In [ ]:
parametros = {"n_estimators": 200, "max_depth": 5, "min_samples_leaf": 50, "max_features": "sqrt",
              "class_weight": None, "random_state": 42, "n_jobs": -1}

# Umbral de alerta: se elige en las últimas 3 temporadas de entrenamiento, maximizando el F1
ajuste = train[train["temporada"] <= "2018-2019"]
validacion = train[train["temporada"] >= "2019-2020"]
modelo = RandomForestClassifier(**parametros).fit(ajuste[variables], ajuste["dia_critico"])
prob_val = modelo.predict_proba(validacion[variables])[:, 1]
precision, recall, cortes_pr = precision_recall_curve(validacion["dia_critico"], prob_val)
f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-9)
umbral = round(float(cortes_pr[np.argmax(f1)]), 3)

# Modelo final entrenado con todo el entrenamiento y evaluado en prueba
modelo = RandomForestClassifier(**parametros).fit(X_train, y_train)
prob_train = modelo.predict_proba(X_train)[:, 1]
prob_test = modelo.predict_proba(X_test)[:, 1]
alerta = (prob_test >= umbral).astype(int)

print("Umbral de alerta:", umbral)
print("AUC entrenamiento:", round(roc_auc_score(y_train, prob_train), 3))
print("AUC prueba:", round(roc_auc_score(y_test, prob_test), 3), "(meta ≥ 0,75)")
print("PR AUC prueba:", round(average_precision_score(y_test, prob_test), 3), "| azar:", round(y_test.mean(), 3))
print("Recall:", round(recall_score(y_test, alerta), 3), "| Precision:", round(precision_score(y_test, alerta), 3),
      "| F1:", round(f1_score(y_test, alerta), 3))

### 3.1 Comparación con las reglas de CONAF

- **Botón Rojo (aproximado):** probabilidad de ignición ≥ 70% y viento ≥ 20 km/h en la tarde.
- **30-30-30:** 2 o más condiciones entre temperatura > 30 °C, humedad < 30% y viento > 30 km/h.

In [ ]:
def resumen(pred, nombre):
    return {"Criterio": nombre,
            "Días alertados": int(pred.sum()),
            "Recall": round(recall_score(y_test, pred), 3),
            "Precision": round(precision_score(y_test, pred, zero_division=0), 3),
            "F1": round(f1_score(y_test, pred), 3)}

comparacion = pd.DataFrame([
    resumen(test["boton_rojo_aprox"].values, "Botón Rojo (aprox.)"),
    resumen((test["indice_30_30_30"] >= 2).astype(int).values, "30-30-30 (2 o más)"),
    resumen(alerta, "Modelo Random Forest"),
])
comparacion

### 3.2 Niveles de riesgo

La probabilidad se traduce en 4 niveles con los percentiles 50, 80 y 95 de las probabilidades de entrenamiento. Cada nivel tiene una acción sugerida, que se valida con el socio comunitario.

In [ ]:
cortes = np.quantile(prob_train, [0.5, 0.8, 0.95])
niveles = ["Bajo", "Moderado", "Alto", "Extremo"]
acciones = {"Bajo": "Operación normal",
            "Moderado": "Difusión preventiva a visitantes y vecinos",
            "Alto": "Reforzar patrullajes en los sectores prioritarios",
            "Extremo": "Evaluar restricción de actividades o cierre, en coordinación con CONAF"}

riesgo = test[["fecha", "dia_critico"]].copy()
riesgo["prob"] = prob_test
riesgo["nivel"] = pd.cut(riesgo["prob"], [-np.inf, *cortes, np.inf], labels=niveles)

tabla_niveles = riesgo.groupby("nivel", observed=False).agg(dias=("dia_critico", "size"), criticos=("dia_critico", "sum"))
tabla_niveles["% días críticos en el nivel"] = (tabla_niveles["criticos"] / tabla_niveles["dias"] * 100).round(1)
tabla_niveles["% del total de días críticos"] = (tabla_niveles["criticos"] / riesgo["dia_critico"].sum() * 100).round(1)
tabla_niveles["Acción"] = [acciones[n] for n in tabla_niveles.index]
print("Cortes de probabilidad:", cortes.round(3))
tabla_niveles

## 4. Resultado del modelo diario

- Cumple la meta: AUC de 0,85 en las temporadas de prueba, sin sobreajuste (AUC de entrenamiento 0,86).
- Supera a las reglas de CONAF: detecta el 61% de los días críticos, frente al 42% de la regla 30-30-30 y el 8% del Botón Rojo, con mejor precisión.
- Los niveles funcionan: la tasa de días críticos sube de 1% (Bajo) a 46% (Extremo). Alto y Extremo reúnen el 27% de los días, pero el 70% de los días críticos.
- Limitación: uno de cada dos días con alerta no termina en incendio, y en marzo y abril el modelo casi no alerta.

## 5. Variables que explican los incendios (objetivo 1)

**Importancia por permutación:** se desordena una variable a la vez en prueba y se mide cuánto cae el AUC.

In [ ]:
importancia = permutation_importance(modelo, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=42, n_jobs=-1)
ranking = pd.DataFrame({"Variable": variables, "Caída AUC": importancia.importances_mean})
ranking = ranking.sort_values("Caída AUC", ascending=False).reset_index(drop=True)
ranking.index += 1

ranking.head(15).iloc[::-1].plot(x="Variable", y="Caída AUC", kind="barh", figsize=(8, 6), color="#4C72B0", legend=False)
plt.title("Importancia por permutación (prueba 2022-2025)")
plt.show()

print("Posición de HCFM:", ranking.index[ranking["Variable"] == "hcfm_tarde_min"][0],
      "| Posición de la probabilidad de ignición:", ranking.index[ranking["Variable"] == "pi_tarde_max"][0])
ranking.head(15)

- La variable más importante es el promedio de incendios en la Región Metropolitana de los 7 días previos: refleja periodos de alta actividad regional.
- Le siguen las variables de sequedad y calor: evapotranspiración, radiación, temperatura y humedad de la tarde.
- La HCFM y la probabilidad de ignición (los índices de CONAF) aportan, pero menos que la evapotranspiración, porque miden lo mismo de otra forma.
- Este ranking cumple la meta del objetivo 1 y se revisará con el socio comunitario.

## 6. Mapa de prioridad (¿dónde?)

El modelo diario dice **cuándo** tener más cuidado. Para saber **dónde**, que fue lo que más valoró el socio, se divide el entorno en celdas de 2 x 2 km.

- **Una fila por celda y temporada** (484 celdas x 22 temporadas).
- **Objetivo:** 1 si en la celda hubo al menos un incendio esa temporada.
- **Variables:** relieve, vegetación, caminos, edificaciones, distancia al parque e **incendios de temporadas anteriores** en la celda y en sus vecinas (solo temporadas pasadas, sin usar información futura).
- Misma partición temporal: entrenamiento hasta 2021-22 y prueba 2022-23 a 2024-25.

In [ ]:
celdas["celda"] = celdas["i"].astype(str) + "," + celdas["j"].astype(str)
incendios["celda"] = incendios["celda_i"].astype(str) + "," + incendios["celda_j"].astype(str)
temporadas = sorted(incendios["temporada"].unique())

conteo = incendios.groupby(["celda", "temporada"]).size().unstack(fill_value=0)
conteo = conteo.reindex(index=celdas["celda"], columns=temporadas, fill_value=0)

# Celdas vecinas (las 8 que rodean a cada celda)
posicion = {c: k for k, c in enumerate(celdas["celda"])}
vecinas = []
for i, j in zip(celdas["i"], celdas["j"]):
    lista = [posicion[f"{i + a},{j + b}"] for a in (-1, 0, 1) for b in (-1, 0, 1)
             if (a or b) and f"{i + a},{j + b}" in posicion]
    vecinas.append(lista)

def historial(hasta):
    # promedio de incendios por temporada en la celda y en sus vecinas, en las temporadas anteriores a 'hasta'
    previas = [t for t in temporadas if t < hasta]
    prom = conteo[previas].mean(axis=1).values
    prom_vecinas = np.array([prom[v].mean() for v in vecinas])
    return prom, prom_vecinas

filas = []
for t in temporadas[1:] + ["2025-2026"]:
    prom, prom_vecinas = historial(t)
    tabla = celdas.copy()
    tabla["temporada"] = t
    tabla["incendios_previos"] = prom
    tabla["incendios_previos_vecinas"] = prom_vecinas
    tabla["hubo_incendio"] = (conteo[t].values > 0).astype(int) if t in conteo.columns else np.nan
    filas.append(tabla)
panel = pd.concat(filas, ignore_index=True)

var_celda = ["dist_parque_km", "elev_media_m", "elev_desv_m", "pendiente_media_g", "pendiente_max_g", "frac_ladera_norte",
             "ndvi_verano", "km_vial_principal", "km_vial_secundario", "km_calle_local", "km_huella_sendero", "edificios",
             "incendios_previos", "incendios_previos_vecinas"]
train_c = panel[panel["temporada"] <= "2021-2022"]
test_c = panel[panel["temporada"].between("2022-2023", "2024-2025")]
print("Entrenamiento:", train_c.shape, "| celdas-temporada con incendio:", round(train_c["hubo_incendio"].mean(), 3))
print("Prueba:", test_c.shape, "| celdas-temporada con incendio:", round(test_c["hubo_incendio"].mean(), 3))

In [ ]:
modelo_celdas = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1)
modelo_celdas.fit(train_c[var_celda], train_c["hubo_incendio"])
test_c = test_c.copy()
test_c["prob"] = modelo_celdas.predict_proba(test_c[var_celda])[:, 1]
print("AUC prueba (celda-temporada):", round(roc_auc_score(test_c["hubo_incendio"], test_c["prob"]), 3))

# Meta: el 20% de las celdas de mayor riesgo debe concentrar al menos el 50% de los incendios de prueba
riesgo_celda = test_c.groupby("celda")["prob"].mean()
incendios_test = incendios[incendios["temporada"] >= "2022-2023"].groupby("celda").size()
incendios_test = incendios_test.reindex(riesgo_celda.index, fill_value=0)
top20 = riesgo_celda.sort_values(ascending=False).index[: int(len(riesgo_celda) * 0.2)]
print("Incendios de prueba en el 20% de celdas de mayor riesgo:",
      round(incendios_test[top20].sum() / incendios_test.sum() * 100, 1), "% (meta ≥ 50%)")

importancia_celdas = pd.Series(modelo_celdas.feature_importances_, index=var_celda).sort_values(ascending=False)
importancia_celdas.round(3)

In [ ]:
mapa = celdas.set_index("celda").loc[riesgo_celda.index, ["lat", "lon"]]
mapa["riesgo"] = riesgo_celda
mapa["prioridad"] = pd.qcut(mapa["riesgo"], [0, 0.5, 0.8, 0.95, 1], labels=["Baja", "Media", "Alta", "Muy alta"])
inc_test = incendios[incendios["temporada"] >= "2022-2023"]

plt.figure(figsize=(8, 7))
plt.scatter(mapa["lon"], mapa["lat"], c=mapa["riesgo"], cmap="YlOrRd", marker="s", s=45)
plt.colorbar(label="probabilidad de incendio en la temporada")
plt.scatter(inc_test["lon"], inc_test["lat"], s=6, c="k", label="incendios reales 2022-2025")
plt.scatter([-70.45], [-33.75], marker="*", s=300, c="green", edgecolor="k", label="PN Río Clarillo")
plt.title("Mapa de prioridad (prueba 2022-2025)")
plt.legend(loc="lower left")
plt.show()

resumen_prioridad = mapa.groupby("prioridad", observed=False).agg(celdas=("riesgo", "size"))
resumen_prioridad["incendios reales"] = incendios_test.groupby(mapa["prioridad"], observed=False).sum()
resumen_prioridad

- AUC de 0,90 en prueba. El 20% de las celdas de mayor riesgo concentró el 80% de los incendios de 2022-2025, muy por sobre la meta del 50%. Las celdas de prioridad Baja (la mitad del área) tuvieron solo 1 incendio.
- Lo que más pesa son los incendios previos de la celda y sus vecinas, la cercanía a calles, huellas y edificaciones, y la menor altura: los incendios ocurren donde hay personas.
- Para el boletín, el riesgo de cada sector en un día se obtiene combinando ambos modelos: probabilidad del día x probabilidad de la celda.

## 7. Boletín semanal con pronóstico de 7 días (objetivo 3)

El socio pidió anticipar los próximos 7 días y recibir el boletín por correo. El proceso:

1. Descargar el pronóstico de Open-Meteo para el mismo punto de los datos de entrenamiento.
2. Calcular las mismas variables del modelo diario y estimar el riesgo de cada día.
3. Tomar las celdas de mayor riesgo de la temporada actual (mapa de prioridad), en todo el entorno y dentro de Pirque.
4. Redactar el texto con IA generativa (Gemini) y guardar el boletín en HTML para enviarlo por correo.

**Redacción con IA generativa:**

- Si quien ejecuta tiene la clave `GEMINI_API_KEY` en los *Secrets* de Colab (ícono de llave), se usa **Gemini**.
- Si no la tiene, se usa **Qwen2.5-1.5B-Instruct**, un modelo de código abierto que corre dentro de Colab sin clave. Así cualquier persona puede ejecutar el notebook completo.
- Si ninguno está disponible, se usa un texto de plantilla.

In [ ]:
import requests
import holidays
from datetime import timedelta

hoy = pd.Timestamp.today().normalize()
diarias = ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean", "precipitation_sum",
           "relative_humidity_2m_max", "relative_humidity_2m_min", "relative_humidity_2m_mean", "wind_speed_10m_max",
           "wind_gusts_10m_max", "wind_direction_10m_dominant", "et0_fao_evapotranspiration", "shortwave_radiation_sum"]
horarias = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_gusts_10m", "precipitation",
            "vapour_pressure_deficit", "shortwave_radiation"]

pronostico = requests.get("https://api.open-meteo.com/v1/forecast", timeout=60, params={
    "latitude": -33.72, "longitude": -70.47, "daily": ",".join(diarias), "hourly": ",".join(horarias),
    "past_days": 31, "forecast_days": 7, "timezone": "America/Santiago"}).json()
historico = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=60, params={
    "latitude": -33.72, "longitude": -70.47, "daily": "precipitation_sum", "timezone": "America/Santiago",
    "start_date": str((hoy - timedelta(days=300)).date()), "end_date": str((hoy - timedelta(days=32)).date())}).json()

clima = pd.DataFrame(pronostico["daily"])
clima.columns = ["fecha", "t_max", "t_min", "t_media", "precip_mm", "hr_max", "hr_min", "hr_media",
                 "viento_max_kmh", "rafaga_max_kmh", "viento_dir", "et0_mm", "radiacion_mj"]
clima["fecha"] = pd.to_datetime(clima["fecha"])
hora = pd.DataFrame(pronostico["hourly"])
hora.columns = ["fecha_hora", "temp", "hr", "viento_kmh", "rafaga_kmh", "precip_mm", "vpd_kpa", "radiacion_wm2"]
hora["fecha_hora"] = pd.to_datetime(hora["fecha_hora"])
lluvia_previa = pd.DataFrame({"fecha": pd.to_datetime(historico["daily"]["time"]),
                              "precip_mm": historico["daily"]["precipitation_sum"]})
print("Pronóstico desde", clima["fecha"].min().date(), "hasta", clima["fecha"].max().date())

Mismos cálculos usados para construir el dataset: condiciones de la tarde (14 a 18:59 h) con HCFM y probabilidad de ignición, días sin lluvia, medias móviles, regla 30-30-30 y calendario. Los incendios de los días previos se reemplazan por el promedio histórico del mes, porque no se conocen para los días futuros.

In [ ]:
def hcfm(t, hr):
    # Humedad del combustible fino muerto (ecuaciones de Simard, t en °F)
    t_f = t * 9 / 5 + 32
    emc = np.where(hr < 10, 0.03229 + 0.281073 * hr - 0.000578 * hr * t_f,
          np.where(hr < 50, 2.22749 + 0.160107 * hr - 0.01478 * t_f,
                   21.0606 + 0.005565 * hr ** 2 - 0.00035 * hr * t_f - 0.483199 * hr))
    return 1.03 * emc

def prob_ignicion(t, m):
    q = 144.5 - 0.266 * t - 0.00058 * t ** 2 - 0.01 * t * m + 18.54 * (1 - np.exp(-0.151 * m)) + 6.4 * m
    x = np.clip((400 - q) / 10, 0, None)
    return np.clip(0.000048 * x ** 4.3 / 50 * 100, 0, 100)

# Condiciones de la tarde
hora["hcfm"] = hcfm(hora["temp"], hora["hr"]).round(2)
hora["pi"] = prob_ignicion(hora["temp"], hora["hcfm"]).round(1)
tarde = hora[hora["fecha_hora"].dt.hour.between(14, 18)].copy()
tarde["fecha"] = tarde["fecha_hora"].dt.normalize()
tarde["hora_br"] = ((tarde["pi"] >= 70) & (tarde["rafaga_kmh"] >= 20)).astype(int)
tarde = tarde.groupby("fecha").agg(t_tarde_max=("temp", "max"), hr_tarde_min=("hr", "min"),
                                   viento_tarde_max=("viento_kmh", "max"), rafaga_tarde_max=("rafaga_kmh", "max"),
                                   vpd_tarde_max=("vpd_kpa", "max"), hcfm_tarde_min=("hcfm", "min"),
                                   pi_tarde_max=("pi", "max"), horas_boton_rojo=("hora_br", "sum")).reset_index()
tarde["boton_rojo_aprox"] = (tarde["horas_boton_rojo"] > 0).astype(int)

# Variables diarias
x = clima.sort_values("fecha").reset_index(drop=True)
x["amplitud_termica"] = x["t_max"] - x["t_min"]
lluvia = pd.concat([lluvia_previa[lluvia_previa["fecha"] < x["fecha"].min()], x[["fecha", "precip_mm"]]])
llueve = lluvia["precip_mm"] >= 1
lluvia["dias_sin_lluvia"] = (~llueve).groupby(llueve.cumsum()).cumsum()
x = x.merge(lluvia[["fecha", "dias_sin_lluvia"]], on="fecha", how="left")
x["t_max_media3"] = x["t_max"].rolling(3).mean()
x["t_max_media7"] = x["t_max"].rolling(7).mean()
x["hr_min_media3"] = x["hr_min"].rolling(3).mean()
x["et0_media7"] = x["et0_mm"].rolling(7).mean()
x["precip_30d"] = x["precip_mm"].rolling(30).sum()
x = x.merge(tarde, on="fecha", how="left")

# Índices y calendario
x["indice_30_30_30"] = (x["t_tarde_max"] > 30).astype(int) + (x["hr_tarde_min"] < 30).astype(int) + (x["rafaga_tarde_max"] > 30).astype(int)
x["calor_x_sequedad"] = (x["t_tarde_max"] * (100 - x["hr_tarde_min"]) / 100).round(2)
x["dia_semana"] = x["fecha"].dt.dayofweek
x["fin_de_semana"] = (x["dia_semana"] >= 5).astype(int)
x["feriado"] = x["fecha"].dt.date.isin(list(holidays.Chile(years=[hoy.year, hoy.year + 1]).keys())).astype(int)
x["mes"] = x["fecha"].dt.month
inicio = pd.to_datetime(np.where(x["mes"] >= 7, x["fecha"].dt.year, x["fecha"].dt.year - 1).astype(str) + "-07-01")
x["dia_temporada"] = (x["fecha"] - inicio).dt.days
x["tasa_mes"] = x["mes"].map(tasa_mes)
x["mes_sin"] = np.sin(2 * np.pi * x["mes"] / 12)
x["mes_cos"] = np.cos(2 * np.pi * x["mes"] / 12)

# Incendios previos: promedio histórico del mes
promedio_mes = datos.groupby("mes")[["entorno_ayer", "rm_ayer", "entorno_media7_previa", "rm_media7_previa"]].mean()
for col in promedio_mes.columns:
    x[col] = x["mes"].map(promedio_mes[col])

semana = x[x["fecha"] >= hoy].reset_index(drop=True)
print("Valores faltantes en la semana:", semana[variables].isna().sum().sum())

In [ ]:
# Riesgo de cada día: el modelo se reentrena con las 23 temporadas
modelo_operativo = RandomForestClassifier(**parametros).fit(datos[variables], datos["dia_critico"])
semana["prob"] = modelo_operativo.predict_proba(semana[variables])[:, 1]
semana["nivel"] = pd.cut(semana["prob"], [-np.inf, *cortes, np.inf], labels=niveles)
semana["accion"] = semana["nivel"].map(acciones)

dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
tabla_semana = pd.DataFrame({
    "Día": [dias[f.dayofweek] + " " + f.strftime("%d-%m") for f in semana["fecha"]],
    "T° máx (°C)": semana["t_max"].round(1),
    "HR mín tarde (%)": semana["hr_tarde_min"].round(0),
    "Ráfaga tarde (km/h)": semana["rafaga_tarde_max"].round(0),
    "Prob. día crítico (%)": (semana["prob"] * 100).round(0),
    "Nivel": semana["nivel"],
    "Acción sugerida": semana["accion"]})

# Sectores prioritarios de la temporada actual (modelo de celdas con el historial hasta 2024-25)
actual = panel[panel["temporada"] == "2025-2026"].copy()
actual["prob"] = modelo_celdas.predict_proba(actual[var_celda])[:, 1]
cercano = incendios.groupby("celda")["comuna"].agg(lambda s: s.mode()[0])
actual["sector"] = actual["celda"].map(cercano).fillna("Sin incendios previos")
sectores = actual.sort_values("prob", ascending=False).head(10)
tabla_sectores = pd.DataFrame({
    "Sector": sectores["sector"].values,
    "Mapa": ["https://maps.google.com/?q=" + f"{la:.4f},{lo:.4f}" for la, lo in zip(sectores["lat"], sectores["lon"])],
    "Prob. temporada (%)": (sectores["prob"] * 100).round(0).values})
# El socio pidió saber qué lugares de Pirque son más propensos
pirque = actual[actual["sector"] == "Pirque"].sort_values("prob", ascending=False).head(5)
tabla_pirque = pd.DataFrame({
    "Mapa": ["https://maps.google.com/?q=" + f"{la:.4f},{lo:.4f}" for la, lo in zip(pirque["lat"], pirque["lon"])],
    "Prob. temporada (%)": (pirque["prob"] * 100).round(0).values})
display(tabla_semana)
display(tabla_sectores)
tabla_pirque

In [ ]:
prompt = f'''Eres un asistente del equipo de guardaparques del Parque Nacional Río Clarillo (CONAF, Pirque, Chile).
Redacta un boletín semanal de riesgo de incendio forestal en español, claro, sin tecnicismos ni emojis, de 150 a 250 palabras.
Usa solo los datos de las tablas, sin inventar cifras ni lugares.
Incluye: resumen de la semana, días de mayor riesgo y por qué, sectores donde concentrar patrullajes y recomendaciones.
Termina indicando que es una segunda opinión que complementa el pronóstico oficial de CONAF.

Riesgo por día:
{tabla_semana.to_string(index=False)}

Sectores prioritarios del entorno:
{tabla_sectores.drop(columns="Mapa").to_string(index=False)}

Sectores de mayor riesgo dentro de Pirque (probabilidad de incendio en la temporada):
{tabla_pirque.drop(columns="Mapa").to_string(index=False)}
'''

texto = None

# Opción 1: Gemini, si quien ejecuta tiene su clave GEMINI_API_KEY en los Secrets de Colab
try:
    from google.colab import userdata
    from google import genai
    import time
    cliente = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
    for intento in range(3):              # si el servicio está saturado, reintenta hasta 3 veces
        try:
            texto = cliente.models.generate_content(model="gemini-3.8-flash", contents=prompt).text
            print("Texto redactado con Gemini\n")
            break
        except Exception as error:
            if intento == 2:
                raise
            print("Gemini ocupado, reintentando en 20 segundos...")
            time.sleep(20)
except Exception as error:
    print("No se usó Gemini:", error)

# Opción 2: modelo de código abierto Qwen2.5 (Apache 2.0), corre en Colab sin clave
if texto is None:
    try:
        import torch
        from transformers import pipeline
        print("Usando el modelo abierto Qwen2.5-1.5B-Instruct (la primera vez se descarga, 1 a 2 minutos)...")
        generador = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
                             device=0 if torch.cuda.is_available() else -1)
        salida = generador([{"role": "user", "content": prompt}], max_new_tokens=450, do_sample=False)
        texto = salida[0]["generated_text"][-1]["content"]
        print("Texto redactado con Qwen2.5 (revisar las cifras antes de enviar)\n")
    except Exception as error:
        print("No se usó el modelo abierto:", error)

# Opción 3: texto de plantilla, sin IA
if texto is None:
    altos = tabla_semana[tabla_semana["Nivel"].isin(["Alto", "Extremo"])]["Día"]
    texto = ("Boletín de riesgo de incendio para la semana del " + hoy.strftime("%d-%m-%Y") + ".\n"
             "Días de riesgo alto o extremo: " + (", ".join(altos) if len(altos) else "ninguno") + ".\n"
             "Sectores prioritarios: " + ", ".join(tabla_sectores["Sector"].unique()[:5]) + ".\n"
             "Este boletín es una segunda opinión que complementa el pronóstico oficial de CONAF.")
print(texto)

html = ("<h2>Clarillo Predice - Boletín de riesgo de incendio</h2>"
        + "".join("<p>" + p + "</p>" for p in texto.split("\n") if p.strip())
        + "<h3>Riesgo por día</h3>" + tabla_semana.to_html(index=False)
        + "<h3>Sectores prioritarios</h3>" + tabla_sectores.to_html(index=False, render_links=True)
        + "<h3>Sectores de mayor riesgo en Pirque</h3>" + tabla_pirque.to_html(index=False, render_links=True))
archivo = "boletin_" + hoy.strftime("%Y-%m-%d") + ".html"
with open(archivo, "w", encoding="utf-8") as f:
    f.write(html)
print("Boletín guardado:", archivo)

El archivo HTML se adjunta o se pega en el correo al equipo del parque. Cada semana se registra el nivel del boletín junto al nivel de la herramienta de CONAF, para compararlas durante la prueba piloto.

## 8. Conclusiones

**Cumplimiento de los objetivos**

| Objetivo | Meta | Resultado |
|---|---|---|
| 1. Variables que explican los incendios | ranking de 10 o más variables | ranking de 39 variables; destacan la actividad regional reciente, la evapotranspiración, la radiación y el calor de la tarde |
| 2a. Modelo de días críticos | AUC ≥ 0,75 y superar a CONAF | AUC 0,85; detecta el 61% de los días críticos (30-30-30: 42%; Botón Rojo: 8%) |
| 2b. Mapa de prioridad | 20% del área con 50% o más de los incendios | el 20% de las celdas de mayor riesgo concentró el 80% de los incendios de prueba (AUC 0,90) |
| 3. Boletín semanal | boletín con pronóstico de 7 días por correo | proceso listo; falta la prueba piloto con el socio |

**Limitaciones**

- La mayoría de los incendios son causados por personas, y el clima tiene un techo para predecirlos.
- El modelo se entrenó con clima de reanálisis y en el boletín se aplica a un pronóstico, que pierde precisión hacia el día 7.
- Los incendios de los días previos se reemplazan por el promedio del mes en el boletín.
- El mapa usa la red de caminos actual de OpenStreetMap.

**Próximos pasos:** enviar los boletines de la prueba piloto, comparar con la herramienta de CONAF y medir la utilidad con el socio (meta ≥ 4 de 5).